In [1]:
import math
from typing import List, Tuple

def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    words = text.split()
    if not words:
        return []
    
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks

def cosine_similarity(v1: List[float], v2: List[float]) -> float:
    dot_product = sum(a * b for a, b in zip(v1, v2))
    norm_v1 = math.sqrt(sum(a * a for a in v1))
    norm_v2 = math.sqrt(sum(b * b for b in v2))
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
    return dot_product / (norm_v1 * norm_v2)

def top_k_retrieval(query_emb: List[float], doc_embs: List[List[float]], k: int = 2) -> List[Tuple[int, float]]:
    scores = []
    for idx, doc_emb in enumerate(doc_embs):
        score = cosine_similarity(query_emb, doc_emb)
        scores.append((idx, score))
    
    # Sort descending by score
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

# Quick Sanity Check
if __name__ == "__main__":
    sample_text = "Humana healthcare platforms require reliable data pipelines and structured insights from unstructured logs."
    print("Chunks:", chunk_text(sample_text, chunk_size=5, overlap=2))
    
    q_vec = [1.0, 0.0, 0.5]
    d_vecs = [[0.9, 0.1, 0.4], [0.0, 1.0, 0.0], [0.8, 0.2, 0.6]]
    print("Top Matches (Index, Score):", top_k_retrieval(q_vec, d_vecs, k=2))

Chunks: ['Humana healthcare platforms require reliable', 'require reliable data pipelines and', 'pipelines and structured insights from', 'insights from unstructured logs.']
Top Matches (Index, Score): [(0, 0.9938586931957765), (2, 0.9647638212377322)]


In [18]:
# Write a Python class/function that receives a mock JSON string from an LLM response, validates it using Pydantic, 
# handles JSON parsing errors or missing fields gracefully, and enforces fallback default values.

import json
from typing import Optional, List
from pydantic import BaseModel, Field, ValidationError

class MedicalRecordInsight(BaseModel):
    member_id: str
    risk_score: float = Field(..., ge=0.0, le=1.0, description="Risk score between 0 and 1")
    key_conditions: List[str] = Field(default_factory=list)
    action_required: bool = False

def parse_llm_response(raw_llm_output: str) -> Optional[MedicalRecordInsight]:
    try:
        # Sanitize LLM potential markdown code blocks
        clean_json = raw_llm_output.strip()
        if clean_json.startswith("json"):
            clean_json = clean_json.replace("json", "").replace("```", "").strip()

        print('clean_json',clean_json)    
        
        data = json.loads(clean_json)
        print('clean_json22',clean_json)    
        validated_insight = MedicalRecordInsight(**data)
        return validated_insight
    
    except (json.JSONDecodeError, ValidationError) as e:
        print(f"Validation/Parsing Error: {e}")
        # Return fallback or log issue
        return None

# Quick Sanity Check
mock_llm_text = """json
{
    "member_id": "MEM-99281",
    "risk_score": 0.85,
    "key_conditions": ["Hypertension", "Diabetes"],
    "action_required": true
}
"""
result = parse_llm_response(mock_llm_text)
print("Parsed Result:", result)


clean_json {
    "member_id": "MEM-99281",
    "risk_score": 0.85,
    "key_conditions": ["Hypertension", "Diabetes"],
    "action_required": true
}
clean_json22 {
    "member_id": "MEM-99281",
    "risk_score": 0.85,
    "key_conditions": ["Hypertension", "Diabetes"],
    "action_required": true
}
Parsed Result: member_id='MEM-99281' risk_score=0.85 key_conditions=['Hypertension', 'Diabetes'] action_required=True


In [27]:
# Enterprise AI engineers building production platforms must handle API rate-limits or transient backend failures gracefully
import time
import random
from functools import wraps
def with_backoff1(max_retries: int = 10, initial_delay: float = 1.0, backoff_factor: float = 2.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            delay = initial_delay
            for attempt in range(1, max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"[Attempt {attempt}/{max_retries}] Failed with error: {e}")
                    if attempt == max_retries:
                        raise e
                    time.sleep(delay)
                    delay *= backoff_factor
        return wrapper
    return decorator

# Example API call simulation
@with_backoff1(max_retries=6, initial_delay=0.5)
def call_llm_api(prompt: str) -> str:
    # Simulate random transient network error
    if random.random() < 0.7:
        raise ConnectionError("503 Service Unavailable / Rate Limit Exceeded")
    return f"Response for prompt: {prompt}"

# Test execution
try:
    response = call_llm_api("Summarize patient chart")
    print("API Success:", response)
except Exception as final_err:
    print("Final Failure:", final_err)




[Attempt 1/6] Failed with error: 503 Service Unavailable / Rate Limit Exceeded
[Attempt 2/6] Failed with error: 503 Service Unavailable / Rate Limit Exceeded
[Attempt 3/6] Failed with error: 503 Service Unavailable / Rate Limit Exceeded
[Attempt 4/6] Failed with error: 503 Service Unavailable / Rate Limit Exceeded
[Attempt 5/6] Failed with error: 503 Service Unavailable / Rate Limit Exceeded
[Attempt 6/6] Failed with error: 503 Service Unavailable / Rate Limit Exceeded
Final Failure: 503 Service Unavailable / Rate Limit Exceeded


In [28]:
# Write a function hybrid_rerank that takes a list of candidate document texts, their dense cosine similarity scores relative to a query, 
# and the query string. Compute a keyword matching score based on token overlap (Jaccard similarity) and blend it with the vector score using a 
# weighting factor $\alpha$ to return re-ranked results

from typing import List, Dict, Any

def jaccard_similarity(text1: str, text2: str) -> float:
    set1 = set(text1.lower().split())
    set2 = set(text2.lower().split())
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    return len(intersection) / len(union) if union else 0.0

def hybrid_rerank(
    query: str,
    documents: List[str],
    vector_scores: List[float],
    alpha: float = 0.6 # Weight for vector score; (1-alpha) for keyword
) -> List[Dict[str, Any]]:
    
    reranked_results = []
    
    for doc, vec_score in zip(documents, vector_scores):
        kw_score = jaccard_similarity(query, doc)
        # Combined hybrid score computation
        final_score = (alpha * vec_score) + ((1 - alpha) * kw_score)
        
        reranked_results.append({
            "doc": doc,
            "hybrid_score": round(final_score, 4),
            "vector_score": vec_score,
            "keyword_score": round(kw_score, 4)
        })
    
    # Sort descending by hybrid score
    reranked_results.sort(key=lambda x: x["hybrid_score"], reverse=True)
    return reranked_results

# Sanity Check
query = "patient eligibility for ICD-10 E11.9 diabetes coverage"
docs = [
    "General coverage policy regarding adult inpatient care",
    "Eligibility and benefits breakdown for diabetes care code E11.9",
    "Diabetes management plan for chronic conditions"
]
vec_scores = [0.82, 0.75, 0.88] # Pure semantic score preferred doc 3

results = hybrid_rerank(query, docs, vec_scores, alpha=0.5)
for r in results:
    print(r)

{'doc': 'Eligibility and benefits breakdown for diabetes care code E11.9', 'hybrid_score': 0.5417, 'vector_score': 0.75, 'keyword_score': 0.3333}
{'doc': 'Diabetes management plan for chronic conditions', 'hybrid_score': 0.5309, 'vector_score': 0.88, 'keyword_score': 0.1818}
{'doc': 'General coverage policy regarding adult inpatient care', 'hybrid_score': 0.4485, 'vector_score': 0.82, 'keyword_score': 0.0769}


In [35]:
# Write a pure Python function calculate_groundedness_score that calculates claim-level precision. 
#     Given a generated response broken down into sentences and a retrieved context string, 
#     calculate what fraction of response claim sentences contain direct keyword evidence from the context.


import re
from typing import List, Dict

def calculate_groundedness_score(generated_answer: str, retrieved_context: str) -> Dict[str, Any]:
    # Split answer into sentences
    sentences = [s.strip() for s  in re.split(r'[.!?]', generated_answer) if s.strip()]
    if not sentences:
        return {"groundedness_score": 0.0, "unsupported_claims": []}

    context_words = set(retrieved_context.lower().split())
    grounded_count = 0
    unsupported_claims = []

    for sentence in sentences:
        # Extract meaningful tokens (length > 3) to exclude stop words
        tokens = [w.lower() for w in re.findall(r'\b\w+\b', sentence) if len(w) > 3]
        if not tokens:
            continue
        
        # Check token overlap ratio against context
        matched_tokens = [t for t in tokens if t in context_words]
        overlap_ratio = len(matched_tokens) / len(tokens)
        
        # Threshold: if >50% of key tokens are present in context, mark grounded
        if overlap_ratio >= 0.5:
            grounded_count += 1
        else:
            unsupported_claims.append(sentence)

    score = grounded_count / len(sentences)
    return {
        "groundedness_score": round(score, 2),
        "total_claims": len(sentences),
        "grounded_claims": grounded_count,
        "unsupported_claims": unsupported_claims
    }

# Sanity Check
context = "Member John Doe is enrolled in Medicare Advantage Plan ID 8831. Deductible is $500."
answer = "John Doe is enrolled in Medicare Advantage Plan ID 8831. His primary care physician is Dr. Smith."

eval_result = calculate_groundedness_score(answer, context)
print("Evaluation Result:", eval_result)

Evaluation Result: {'groundedness_score': 0.33, 'total_claims': 3, 'grounded_claims': 1, 'unsupported_claims': ['His primary care physician is Dr', 'Smith']}
